In [1]:
import numpy as np
import pandas as pd
import keras
from keras.layers import Dense
import os
import pandas
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/nyc-taxi-traffic/dataset.csv


This notebook explains implementation of TCN using Keras. Implementation with optimization. We'd try to predict traffic for 1 day.

# Prerequisite

## Install TCN
There is not yet support for TCN in Keras api. I am using keras-tcn library https://pypi.org/project/keras-tcn/
> pip install keras-tcn

In [2]:
pip install keras-tcn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 20.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 48.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - done
  Created wheel for wrapt: filename=wrapt-1.12.1-cp37-cp37m-linux_x86_64.whl size=77041 sha256=034988a50050a5968c9062b6848814a83ebe1f65f781fc80b624fe5d1f22dcfd
  Stored in directory: /root/.cache/pip/wheels/62/76/4c/aa25851149f3f6d9785f6c869387ad82b3fd37582fa8147ac6
Successfully built wrapt
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.14.1
    Uninstalling wrapt-1.14.1:
      Successfully uninstalled wrapt-1.14.1
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.1.1
    Uninstalling typing_extensions-4.1.1:
      Successfully uninstalled typing_extensions-4.1.1
  Attempting uninstall: six
    Found existing installation: six 1.16.0
   

# Run TCN

#### TCN model with Bayesian Optimization

In [3]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense
from keras_tuner import HyperModel
import kerastuner as kt

from tcn import TCN

def tcnModel(
    trainData, 
    validateData,
    numOfObservedRecords: int,
	numOfPredictingRecords: int
):  
    hyperModel = TCNModel(
        numOfObservedRecords=numOfObservedRecords,
        numOfPredictingRecords=numOfPredictingRecords,
        numOfFeatures=trainData.shape[2])
    bayesianTuner = kt.tuners.BayesianOptimization(
        hyperModel,
        objective="mse",
        max_trials=3,
        project_name='kerastuner_bayesian_poc',
        executions_per_trial=5,
        overwrite=True)
    bayesianTuner.search(trainData, validateData,
                         epochs=100, validation_split=0.2, verbose=0)
    return bayesianTuner.get_best_models(num_models=1)[0]  # get best model

class TCNModel(HyperModel):

    def __init__(self, numOfObservedRecords, numOfPredictingRecords, numOfFeatures):
        self.numOfObservedRecords = numOfObservedRecords
        self.numOfPredictingRecords = numOfPredictingRecords
        self.numOfFeatures = numOfFeatures

    def build(self, params):
        model = keras.Sequential()
        model.add(TCN(
            input_shape=(self.numOfObservedRecords, 1),
            kernel_size=params.Int('units', min_value=2, max_value=8, step=1),
            use_skip_connections=params.Boolean('use_skip_connections'),
            use_batch_norm=False,
            use_weight_norm=False,
            use_layer_norm=True,
            dropout_rate=params.Float('drop_out', 0, 0.5, 0.1),
            nb_filters=params.Int('units', min_value=32, max_value=512, step=32),
        ))
        model.add(Dense(
            self.numOfPredictingRecords,
            activation=params.Choice(
                'dense_activation',
                values=['relu', 'tanh', 'sigmoid'],
                default='relu'
        )))
        model.compile(
            loss='mse', metrics=['mse'],
            optimizer=tf.keras.optimizers.Adam(params.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])))
        return model

/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:5: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  """


In [4]:
from sklearn.preprocessing import MinMaxScaler

def scale3DArray(arr: np.ndarray, scaler: any = MinMaxScaler(feature_range=(-1, 1))) -> np.ndarray:
    scaledArr = np.reshape(arr, (arr.shape[0], arr.shape[1] * arr.shape[2]))
    scaledArr = scaler.fit_transform(scaledArr)
    scaledArr = np.reshape(scaledArr, tuple(arr.shape))
    return scaledArr


def scale3DArrayReturningScaler(arr: np.ndarray, scaler: any = MinMaxScaler(feature_range=(-1, 1))):
    scaledArr = np.reshape(arr, (arr.shape[0], arr.shape[1] * arr.shape[2]))
    scaledArr = scaler.fit_transform(scaledArr)
    scaledArr = np.reshape(scaledArr, tuple(arr.shape))
    return scaledArr, scaler

Run model

In [5]:
data = pandas.read_csv('../input/nyc-taxi-traffic/dataset.csv')
print(data.head())
print(len(data))
data = list(data['value'])[:10200]
print(len(data))

   Unnamed: 0            timestamp  value
0           0  2014-07-01 00:00:00  10844
1           1  2014-07-01 00:30:00   8127
2           2  2014-07-01 01:00:00   6210
3           3  2014-07-01 01:30:00   4656
4           4  2014-07-01 02:00:00   3820
10320
10200


In [6]:
data_train = data[: int(len(data) * 0.8)]
data_test = data[int(len(data) * 0.8):]

print("len train ", len(data_train))
print("len test ", len(data_test))

O_data_test = data_test

len train  8160
len test  2040


In [7]:
minmaxSc = MinMaxScaler()

data_train = np.array(data_train)
print(data_train.shape)
data_train = np.reshape(data_train, (int(data_train.shape[0] / 120), 120))
print(data_train.shape)

data_train = minmaxSc.fit_transform(data_train)
print(data_train.shape)

train, validate = np.array_split(data_train, [72], 1)
print(train.shape)
print(validate.shape)

train = np.reshape(train, (train.shape[0], train.shape[1], 1))
validate = np.reshape(validate, (validate.shape[0], validate.shape[1], 1))
print(train.shape)
print(validate.shape)
    

(8160,)
(68, 120)
(68, 120)
(68, 72)
(68, 48)
(68, 72, 1)
(68, 48, 1)


In [8]:
model = tcnModel(train, validate, 72, 48)

2022-09-21 14:44:26.855816: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.
2022-09-21 14:44:28.826972: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


In [9]:
data_test = np.array(data_test)
print(data_test.shape)
data_test = np.reshape(data_test, (int(data_test.shape[0] / 120), 120))
print(data_test.shape)

data_test = minmaxSc.transform(data_test)
print(data_test.shape)

test, validate = np.array_split(data_test, [72], 1)
print(test.shape)
print(validate.shape)

test = np.reshape(test, (test.shape[0], test.shape[1], 1))
validate = np.reshape(validate, (validate.shape[0], validate.shape[1], 1))
print(test.shape)
print(validate.shape)

(2040,)
(17, 120)
(17, 120)
(17, 72)
(17, 48)
(17, 72, 1)
(17, 48, 1)


In [10]:
test_predict = np.array(model.predict(test))
print(test_predict.shape)

test_predict = np.reshape(test_predict, (test_predict.shape[0], test_predict.shape[1], 1))
print(test_predict.shape)

test_predict_full = np.concatenate((test, test_predict), axis=1)
print(test_predict_full.shape)

test_predict_full = np.squeeze(test_predict_full)
print(test_predict_full.shape)

test_predict_full = minmaxSc.inverse_transform(test_predict_full)

test_predict_full = list(test_predict_full.flatten())
print(len(test_predict_full))

(17, 48)
(17, 48, 1)
(17, 120, 1)
(17, 120)
2040


In [11]:
from sklearn.metrics import mean_squared_error, r2_score
r2 = round(r2_score(O_data_test, test_predict_full), 3)
rmse = round(np.sqrt(mean_squared_error(O_data_test, test_predict_full)), 3)

print(r2)
print(rmse)

0.759
3462.071
